# Choosing the Number of Clusters (K): Complete Guide

## 1. Introduction: The K-Means Dilemma
- In unsupervised learning, we don't have labeled target variables. This makes evaluating a model inherently subjective because there is no "ground truth" to compare against.
- When using partitioning algorithms like K-Means, the most critical hyperparameter is 'K' (the number of clusters).
- **Under-segmentation (K is too low):** You lump distinct groups together, losing valuable granular insights. You treat varied populations as a single monolith.
- **Over-segmentation (K is too high):** You split natural groups arbitrarily, creating artificial distinctions that aren't actionable for business strategy.
- In this notebook, we will explore mathematical heuristics to find the "optimal" K, primarily focusing on the **Elbow Method** and the **Silhouette Score**.

In [ ]:
# 2. Setup and Required Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set display options and plot style
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for perfect reproducibility
np.random.seed(42)
print("Libraries successfully imported and environment ready!")

## 3. Data Creation: Retail Customer Segmentation
- We need a dataset where we secretly *know* the true number of clusters to see how our mathematical tests perform.
- We will simulate 400 retail customers based on two features: `Annual_Spend` and `Engagement_Score`.
- We will intentionally create 4 distinct customer archetypes:
  1. Low Spend, Low Engagement (Bargain Hunters)
  2. Low Spend, High Engagement (Brand Fans)
  3. High Spend, Low Engagement (Convenience Shoppers)
  4. High Spend, High Engagement (Loyal VIPs)

In [ ]:
# Generate 4 distinct clusters (100 customers each)
n_per_cluster = 100

# Cluster 1: Bargain Hunters
spend_1 = np.random.normal(20, 5, n_per_cluster)
engage_1 = np.random.normal(20, 5, n_per_cluster)

# Cluster 2: Brand Fans
spend_2 = np.random.normal(20, 5, n_per_cluster)
engage_2 = np.random.normal(80, 5, n_per_cluster)

# Cluster 3: Convenience Shoppers
spend_3 = np.random.normal(80, 5, n_per_cluster)
engage_3 = np.random.normal(20, 5, n_per_cluster)

# Cluster 4: Loyal VIPs
spend_4 = np.random.normal(80, 5, n_per_cluster)
engage_4 = np.random.normal(80, 5, n_per_cluster)

# Combine data
spend = np.concatenate([spend_1, spend_2, spend_3, spend_4])
engage = np.concatenate([engage_1, engage_2, engage_3, engage_4])

df_customers = pd.DataFrame({
    'Annual_Spend': spend,
    'Engagement_Score': engage
})

# Shuffle the dataset
df_customers = df_customers.sample(frac=1, random_state=42).reset_index(drop=True)

# Standardize the data (Always required for distance-based clustering)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_customers)
df_scaled = pd.DataFrame(scaled_data, columns=df_customers.columns)

print("Synthetic Customer Dataset Created and Scaled.")
print(df_scaled.head().round(3))

## 4. Visualizing the Raw Data
- Before running algorithms, we should always visualize the data if possible.
- Because we only have 2 dimensions, we can easily plot this on a scatter plot.
- You will clearly see the 4 natural clusters we engineered.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='Annual_Spend', y='Engagement_Score', data=df_scaled, color='gray', alpha=0.7, s=50)
plt.title('Unlabeled Customer Data (Standardized)', fontsize=15)
plt.xlabel('Annual Spend (Z-Score)')
plt.ylabel('Engagement Score (Z-Score)')
plt.grid(alpha=0.3)
plt.show()

print("Observation: The 4 distinct groups are clearly visible to the human eye.")
print("Our goal is to see if the math agrees with our intuition.")

## 5. Core Concept 1: WCSS (Within-Cluster Sum of Squares)
- To evaluate a cluster's tightness, we use **WCSS** (also known as **Inertia**).
- WCSS = Sum of squared distances between each data point and its assigned cluster centroid.
- **The Rule:** The lower the WCSS, the tighter and more compact the clusters are.
- **The Catch:** If K = N (number of data points), WCSS = 0. Every point is its own cluster. Therefore, we cannot simply minimize WCSS blindly, or we will always choose K=N.

In [ ]:
# We will calculate WCSS for K values from 1 to 10
wcss_values = []
k_range = range(1, 11)

print("--- Calculating WCSS (Inertia) for K=1 to K=10 ---")
for k in k_range:
    # Initialize KMeans
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    # Fit the model
    kmeans.fit(df_scaled)
    # Append the WCSS (inertia_ attribute)
    wcss_values.append(kmeans.inertia_)
    print(f"K = {k:2d} | WCSS = {kmeans.inertia_:.2f}")

print("\nNotice that WCSS always decreases as K increases. It never goes up.")

## 6. Core Concept 2: The Elbow Method
- Since WCSS always decreases, we plot WCSS against K to find the **point of diminishing returns**.
- This plot forms a curve that looks like an arm.
- The "Elbow" is the point where the steep drop levels off into a gentle slope. This represents the optimal K because adding more clusters beyond this point doesn't significantly improve the compactness of the clusters.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss_values, marker='o', linestyle='-', color='purple', linewidth=2, markersize=8)
plt.title('The Elbow Method for Optimal K', fontsize=16)
plt.xlabel('Number of Clusters (K)', fontsize=14)
plt.ylabel('WCSS (Inertia)', fontsize=14)

# Draw a vertical line at the elbow
plt.axvline(x=4, color='red', linestyle='--', label='The Elbow (K=4)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("Analysis:")
print("Going from K=1 to K=2 reduces WCSS massively.")
print("Going from K=3 to K=4 also provides a solid reduction.")
print("But from K=4 to K=5, the line flattens out. The reduction in error is marginal.")
print("Therefore, the 'Elbow' is at K=4.")

## 7. Core Concept 3: The Silhouette Score
- The Elbow method is sometimes ambiguous. A more mathematically rigorous metric is the **Silhouette Score**.
- The Silhouette Score measures how similar an object is to its own cluster compared to other clusters.
- It uses two metrics for a given data point:
  - **a:** The mean distance between a point and all other points in the SAME cluster (Cohesion).
  - **b:** The mean distance between a point and all points in the NEAREST adjacent cluster (Separation).
- **Formula:** Silhouette = (b - a) / max(a, b)
- **Range:** -1 (wrongly assigned) to +1 (perfectly assigned). 0 means overlapping clusters.

In [ ]:
# Calculate Silhouette Scores for K=2 to K=10
# (Silhouette requires at least 2 clusters and less than N clusters)
silhouette_scores = []
k_range_sil = range(2, 11)

print("--- Calculating Silhouette Scores for K=2 to K=10 ---")
for k in k_range_sil:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    cluster_labels = kmeans.fit_predict(df_scaled)
    
    # Calculate the average silhouette score for all points
    sil_score = silhouette_score(df_scaled, cluster_labels)
    silhouette_scores.append(sil_score)
    print(f"K = {k:2d} | Silhouette Score = {sil_score:.4f}")

## 8. Visualizing the Silhouette Scores
- We plot the Silhouette Scores as a bar chart.
- **The Rule:** The optimal K is simply the one with the highest Silhouette Score.

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(k_range_sil, silhouette_scores, color='teal', alpha=0.8)

# Highlight the best score
best_k_idx = np.argmax(silhouette_scores)
bars[best_k_idx].set_color('darkorange')

plt.title('Silhouette Score Evaluation', fontsize=16)
plt.xlabel('Number of Clusters (K)', fontsize=14)
plt.ylabel('Average Silhouette Score', fontsize=14)
plt.xticks(k_range_sil)
plt.axhline(y=silhouette_scores[best_k_idx], color='darkorange', linestyle='--', alpha=0.5)
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"Analysis: The peak of the chart is undeniably at K={k_range_sil[best_k_idx]}.")
print("This corroborates our finding from the Elbow Method.")

## 9. Visual Proof: The Dangers of Under and Over Segmentation
- Let's visually examine what happens when we pick K=2 (Under-segmentation), K=4 (Optimal), and K=6 (Over-segmentation).
- This proves why relying on these mathematical heuristics is crucial.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

test_ks = [2, 4, 6]
titles = ['Under-Segmented (K=2)', 'Optimal (K=4)', 'Over-Segmented (K=6)']
colors = ['magma', 'viridis', 'plasma']

for i, k in enumerate(test_ks):
    kmeans = KMeans(n_clusters=k, init='k-kmeans++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(df_scaled)
    
    sns.scatterplot(x='Annual_Spend', y='Engagement_Score', data=df_scaled, 
                    c=labels, cmap=colors[i], ax=axes[i], s=50, alpha=0.8)
    
    # Plot centroids
    centroids = kmeans.cluster_centers_
    axes[i].scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centroids')
    
    axes[i].set_title(titles[i], fontsize=14)
    axes[i].grid(alpha=0.3)
    if i == 0:
        axes[i].legend()

plt.tight_layout()
plt.show()

print("Visual Breakdown:")
print("- K=2 groups High/Low Spend together, destroying the price-sensitivity insight.")
print("- K=4 perfectly maps to our 4 distinct business archetypes.")
print("- K=6 arbitrarily cuts perfectly cohesive groups in half, creating fake personas.")

## 10. The Reality of "Messy" Data
- The dataset we generated above was exceptionally clean. Real-world business data rarely has such perfect separation.
- When clusters overlap, the Elbow curve won't have a sharp 90-degree kink, and Silhouette scores will be lower overall.
- Let's generate a "Messy" overlapping dataset and see how the metrics respond.

In [ ]:
# Generate overlapping clusters (higher variance/spread)
spend_m1 = np.random.normal(40, 15, 100)
engage_m1 = np.random.normal(40, 15, 100)

spend_m2 = np.random.normal(60, 15, 100)
engage_m2 = np.random.normal(60, 15, 100)

spend_m3 = np.random.normal(40, 15, 100)
engage_m3 = np.random.normal(60, 15, 100)

df_messy = pd.DataFrame({
    'Spend': np.concatenate([spend_m1, spend_m2, spend_m3]),
    'Engage': np.concatenate([engage_m1, engage_m2, engage_m3])
})

scaled_messy = scaler.fit_transform(df_messy)

plt.figure(figsize=(6, 4))
plt.scatter(scaled_messy[:,0], scaled_messy[:,1], color='gray', alpha=0.5)
plt.title('Messy Real-World Data (Overlapping)', fontsize=14)
plt.show()

## 11. Analyzing Messy Data
- We will run both the Elbow Method and Silhouette Score side-by-side on this messy data.
- You will see that finding K becomes a matter of interpreting weak signals.

In [ ]:
wcss_m = []
sil_m = []
k_range_m = range(2, 9)

for k in k_range_m:
    kmeans_m = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_m = kmeans_m.fit_predict(scaled_messy)
    wcss_m.append(kmeans_m.inertia_)
    sil_m.append(silhouette_score(scaled_messy, labels_m))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Messy Elbow
axes[0].plot(k_range_m, wcss_m, marker='o', color='purple')
axes[0].set_title('Messy Elbow Curve', fontsize=14)
axes[0].set_xlabel('K')
axes[0].set_ylabel('WCSS')

# Messy Silhouette
axes[1].bar(k_range_m, sil_m, color='teal')
axes[1].set_title('Messy Silhouette Scores', fontsize=14)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

print("Analysis of Messy Data:")
print("- The Elbow is not sharp. It could arguably be at K=3 or K=4.")
print("- The Silhouette score is highest at K=3, but the score is low (~0.35) compared to our clean data (>0.7).")
print("Business Takeaway: When the math is ambiguous, choose the K that makes the most strategic business sense. Here, K=3 is the safest mathematical bet.")

## 12. Practice Exercises: Patient Diagnostics
**Scenario:** You are analyzing biometric data for 500 patients. You have two scaled features: `Blood_Pressure_Index` and `Heart_Rate_Variability`.
You need to segment these patients to identify risk profiles, but you don't know how many segments exist.

In [ ]:
# Generate Patient Practice Data (5 Clusters)
from sklearn.datasets import make_blobs

X_patients, _ = make_blobs(n_samples=500, centers=5, cluster_std=0.8, random_state=101)
df_patients = pd.DataFrame(X_patients, columns=['BP_Index', 'HRV_Index'])

# Standardize
df_patients_scaled = pd.DataFrame(scaler.fit_transform(df_patients), columns=df_patients.columns)

print("Practice Patient Dataset Created.")
print(df_patients_scaled.head())

### Exercise 1: The Elbow Method
1. Loop through K values from 1 to 10 for the `df_patients_scaled` dataset.
2. Store the inertia values.
3. Plot the Elbow curve. Where is the elbow?

In [ ]:
# --- EXERCISE 1 SOLUTION ---
wcss_patients = []
k_range_pat = range(1, 11)

for k in k_range_pat:
    kmeans_pat = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    kmeans_pat.fit(df_patients_scaled)
    wcss_patients.append(kmeans_pat.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range_pat, wcss_patients, marker='s', color='navy')
plt.title('Exercise 1: Elbow Method for Patients', fontsize=14)
plt.xlabel('K')
plt.ylabel('WCSS')
plt.axvline(x=5, color='red', linestyle='--')
plt.grid(alpha=0.3)
plt.show()

print("Conclusion: The elbow forms prominently at K=5.")

### Exercise 2: The Silhouette Score
1. Loop through K values from 2 to 10 for the patient dataset.
2. Calculate the silhouette score for each.
3. Plot the bar chart. Does it confirm K=5?

In [ ]:
# --- EXERCISE 2 SOLUTION ---
sil_patients = []
k_range_sil_pat = range(2, 11)

for k in k_range_sil_pat:
    kmeans_pat_sil = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_pat = kmeans_pat_sil.fit_predict(df_patients_scaled)
    sil_patients.append(silhouette_score(df_patients_scaled, labels_pat))

plt.figure(figsize=(8, 4))
plt.bar(k_range_sil_pat, sil_patients, color='forestgreen')
plt.title('Exercise 2: Silhouette Scores for Patients', fontsize=14)
plt.xlabel('K')
plt.ylabel('Score')
plt.axhline(y=max(sil_patients), color='red', linestyle='--')
plt.show()

best_k_pat = k_range_sil_pat[np.argmax(sil_patients)]
print(f"Conclusion: The maximum silhouette score is precisely at K={best_k_pat}, confirming the Elbow method.")

## 13. Visualization Gallery: Decision Boundaries
To truly appreciate what K-Means does mathematically when we select K=4, let's visualize the actual decision boundaries (Voronoi regions) it creates in the feature space.

In [ ]:
# Fit final model to original scaled customer data
kmeans_final = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
kmeans_final.fit(df_scaled)

# Create a meshgrid to plot boundaries
h = .02     # step size in the mesh
x_min, x_max = df_scaled['Annual_Spend'].min() - 1, df_scaled['Annual_Spend'].max() + 1
y_min, y_max = df_scaled['Engagement_Score'].min() - 1, df_scaled['Engagement_Score'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict cluster for every point in the meshgrid
Z = kmeans_final.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 8))
plt.imshow(Z, interpolation='nearest', extent=(xx.min(), xx.max(), yy.min(), yy.max()),
           cmap='viridis', aspect='auto', origin='lower', alpha=0.3)

plt.scatter(df_scaled['Annual_Spend'], df_scaled['Engagement_Score'], c=kmeans_final.labels_, cmap='viridis', edgecolors='k', s=50)
plt.scatter(kmeans_final.cluster_centers_[:, 0], kmeans_final.cluster_centers_[:, 1], marker='X', s=250, color='red', label='Centroids')

plt.title('K-Means Decision Boundaries (Voronoi Regions) for K=4', fontsize=16)
plt.xlabel('Annual Spend (Z-Score)', fontsize=12)
plt.ylabel('Engagement Score (Z-Score)', fontsize=12)
plt.legend()
plt.show()

print("The background colors show exactly how K-Means divides the entire mathematical space into 4 territories based on the closest centroid.")

## 14. Summary and Key Takeaways

- **The Challenge:** Selecting K is inherently subjective in unsupervised learning. We rely on heuristics to balance cluster cohesion and segment utility.
- **WCSS (Inertia):** Measures how tightly packed clusters are. It always decreases as K increases, so we cannot simply minimize it.
- **The Elbow Method:** We plot WCSS against K and look for the 'elbow' or kink where adding more clusters yields diminishing returns on error reduction.
- **The Silhouette Score:** A more robust metric measuring separation and cohesion. It ranges from -1 to 1. The K that produces the highest average Silhouette score is statistically the most optimal.
- **Business Context Rules All:** When data is messy and the mathematical signals (Elbow/Silhouette) are ambiguous, you must choose a K that provides the most actionable, interpretable segments for the business strategy.

In [ ]:
print("Module 'Choosing the Number of Clusters (K)' completed successfully.")
print("You are now equipped to mathematically justify your segmentation strategies!")